# 🔬 Model Pipeline: Google Gemma-2-2B-IT (4-bit QLoRA)
### **Jadavpur University Research Internship | Under: Prof. Tohida Rehman**
**Track:** FIRE MixSub-SciHigh Scientific Highlight Summarization
**Model ID:** `google/gemma-2-2b-it`
**Architecture:** CAUSAL (4-bit QLoRA)

---
### 🎯 Strict Uniform Benchmark Constraints (Scientifically Fair)
- **`MAX_SOURCE_LENGTH = 512` tokens**: Uniform input length across all 6 models.
- **`MAX_TARGET_LENGTH = 128` tokens**: Uniform output generation limit across all 6 models.

### 📊 The 3 Non-Redundant Benchmark Pillars Measured:
1. **Overall Quality**: ROUGE-L F1 (↑ Higher is Better) & Target F1 Score (↑)
2. **Hallucination Risk**: Ungrounded Token % (↓ Lower is Better) & Precision Source (% grounded in abstract)
3. **Inference Time**: Wall-Clock Latency per Sample (↓ Lower is Faster) & Throughput (tok/s ↑)

In [ ]:

# CELL 1: Mount Google Drive and configure project paths

from google.colab import drive
import os

drive.mount("/content/drive")

DRIVE_BASE_DIR = "/content/drive/MyDrive/JU_Benchmark_Project"

MODEL_CHECKPOINT_DIR = os.path.join(
    DRIVE_BASE_DIR, "checkpoints", "gemma-2-2b"
)
RESULTS_DIR = os.path.join(DRIVE_BASE_DIR, "results")
DATASET_DIR = os.path.join(DRIVE_BASE_DIR, "Dataset")

os.makedirs(MODEL_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

MODEL_ID = "google/gemma-2-2b-it"

MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 128
MAX_SEQUENCE_LENGTH = MAX_SOURCE_LENGTH + MAX_TARGET_LENGTH

print("Google Drive mounted.")
print("Model:", MODEL_ID)
print("Checkpoint directory:", MODEL_CHECKPOINT_DIR)
print("Results directory:", RESULTS_DIR)
print("Dataset directory:", DATASET_DIR)
print("Input limit:", MAX_SOURCE_LENGTH)
print("Output limit:", MAX_TARGET_LENGTH)


In [ ]:

# CELL 2: Install required libraries

%pip install -q transformers accelerate bitsandbytes peft datasets rouge-score sentencepiece

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not detected. In Colab, select Runtime > Change runtime type > T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2),
    "GB"
)

USE_BF16 = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print("Compute dtype:", COMPUTE_DTYPE)


In [ ]:

# CELL 3: Verify and load FIRE MixSub-SciHigh datasets

import pandas as pd

TRAIN_CSV = os.path.join(
    DATASET_DIR, "MixSub-SciHigh_train_FIRE.csv"
)

VAL_CSV = os.path.join(
    DATASET_DIR, "MixSub-SciHigh_val_FIRE.csv"
)

TEST_CSV = os.path.join(
    DATASET_DIR, "MixSub-SciHigh_test_FIRE(Masked).csv"
)

required_files = {
    "Train": TRAIN_CSV,
    "Validation": VAL_CSV
}

for name, path in required_files.items():
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"{name} dataset not found: {path}\n"
            "Upload the required CSV into the Dataset folder in Google Drive."
        )

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

# Remove rows without training targets
train_df = train_df.dropna(
    subset=["Abstract", "Highlights"]
).reset_index(drop=True)

val_df = val_df.dropna(
    subset=["Abstract", "Highlights"]
).reset_index(drop=True)

# Normalize text columns
for df in [train_df, val_df]:
    df["Abstract"] = df["Abstract"].astype(str).str.strip()
    df["Highlights"] = df["Highlights"].astype(str).str.strip()

    df.drop(
        df[
            (df["Abstract"] == "") |
            (df["Highlights"] == "")
        ].index,
        inplace=True
    )

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))

print("\nTraining example:")
print("Abstract:", train_df.iloc[0]["Abstract"][:300])
print("Highlights:", train_df.iloc[0]["Highlights"][:300])

if os.path.isfile(TEST_CSV):
    test_df = pd.read_csv(TEST_CSV)
    if "Abstract" not in test_df.columns:
        raise ValueError("Masked test CSV must contain an Abstract column.")

    test_df["Abstract"] = test_df["Abstract"].fillna("").astype(str)
    print("\nMasked test samples:", len(test_df))
else:
    test_df = None
    print("\nMasked test file not found. Test prediction will be skipped.")


In [ ]:

# CELL 4: Authenticate and load Gemma-2-2B-IT in 4-bit NF4

from getpass import getpass
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

HF_TOKEN = getpass("Enter your Hugging Face access token: ")

if not HF_TOKEN.strip():
    raise ValueError("A Hugging Face token is required for Gemma.")

login(token=HF_TOKEN)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=COMPUTE_DTYPE,
    token=HF_TOKEN
)

print("Gemma-2-2B-IT loaded successfully.")
print("Quantization: 4-bit NF4")
print("Model device:", model.device)


In [ ]:

# CELL 5: Three-pillar benchmark metric engine

import re
import numpy as np
from collections import Counter
from rouge_score import rouge_scorer

def clean_tokenize(text):
    if not isinstance(text, str):
        return []

    return re.findall(
        r"\b[a-zA-Z0-9]+(?:'[a-zA-Z0-9]+)?\b",
        text.lower()
    )


def get_ngrams(tokens, n=1):
    if len(tokens) < n or n < 1:
        return []

    return [
        tuple(tokens[i:i+n])
        for i in range(len(tokens) - n + 1)
    ]


def compute_pillars_for_sample(gen_text, src_text, tgt_text):
    gen_tokens = clean_tokenize(gen_text)
    src_tokens = clean_tokenize(src_text)
    tgt_tokens = clean_tokenize(tgt_text)

    gen_counts = Counter(get_ngrams(gen_tokens, 1))
    src_counts = Counter(get_ngrams(src_tokens, 1))
    tgt_counts = Counter(get_ngrams(tgt_tokens, 1))

    total_gen = sum(gen_counts.values())
    total_tgt = sum(tgt_counts.values())

    # Pillar 2: Source grounding
    overlap_src = sum((gen_counts & src_counts).values())

    precision_src = (
        overlap_src / total_gen if total_gen > 0 else 0.0
    )

    hallucination_risk = 1.0 - precision_src

    # Pillar 1: Target highlight alignment
    overlap_tgt = sum((gen_counts & tgt_counts).values())

    precision_tgt = (
        overlap_tgt / total_gen if total_gen > 0 else 0.0
    )

    recall_tgt = (
        overlap_tgt / total_tgt if total_tgt > 0 else 0.0
    )

    f1_tgt = (
        2 * precision_tgt * recall_tgt /
        (precision_tgt + recall_tgt)
        if (precision_tgt + recall_tgt) > 0
        else 0.0
    )

    return {
        "hallucination_risk": hallucination_risk,
        "precision_src": precision_src,
        "precision_tgt": precision_tgt,
        "recall_tgt": recall_tgt,
        "f1_tgt": f1_tgt
    }


def evaluate_core_benchmarks(
    predictions,
    sources,
    references,
    latencies=None,
    generated_tokens=None
):
    if not (
        len(predictions) == len(sources) == len(references)
    ):
        raise ValueError("Predictions, sources and references must have equal lengths.")

    if len(predictions) == 0:
        raise ValueError("No samples available for evaluation.")

    scorer = rouge_scorer.RougeScorer(
        ["rougeL"],
        use_stemmer=True
    )

    rouge_f1 = []
    rouge_precision = []
    rouge_recall = []

    for pred, ref in zip(predictions, references):
        score = scorer.score(ref, pred)

        rouge_f1.append(score["rougeL"].fmeasure)
        rouge_precision.append(score["rougeL"].precision)
        rouge_recall.append(score["rougeL"].recall)

    sample_metrics = [
        compute_pillars_for_sample(pred, src, ref)
        for pred, src, ref in zip(predictions, sources, references)
    ]

    results = {
        # Pillar 1: Quality
        "overall_quality_rougeL_f1": float(np.mean(rouge_f1)),
        "rougeL_precision": float(np.mean(rouge_precision)),
        "rougeL_recall": float(np.mean(rouge_recall)),
        "f1_score_target": float(
            np.mean([x["f1_tgt"] for x in sample_metrics])
        ),
        "precision_target": float(
            np.mean([x["precision_tgt"] for x in sample_metrics])
        ),
        "recall_target": float(
            np.mean([x["recall_tgt"] for x in sample_metrics])
        ),

        # Pillar 2: Grounding proxy
        "hallucination_risk_pct": float(
            np.mean([x["hallucination_risk"] for x in sample_metrics])
        ),
        "source_grounding_precision": float(
            np.mean([x["precision_src"] for x in sample_metrics])
        )
    }

    # Pillar 3: Inference speed
    if latencies is not None and len(latencies) > 0:
        results["wall_clock_latency_sec"] = float(np.mean(latencies))
        results["total_wall_clock_sec"] = float(np.sum(latencies))

    if (
        generated_tokens is not None
        and latencies is not None
        and len(latencies) > 0
    ):
        total_tokens = sum(generated_tokens)
        total_time = sum(latencies)

        results["throughput_tokens_per_sec"] = float(
            total_tokens / max(total_time, 1e-8)
        )

    return results


print("Three-pillar benchmark engine loaded.")


In [ ]:

# CELL 7: Evaluate fine-tuned Gemma and generate masked test predictions

import os
import gc
import json
import time
import numpy as np
import pandas as pd
import torch

from peft import PeftModel
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# Number of validation samples.
# Use 100 for a quick benchmark; set to None for the complete validation set.
MAX_EVAL_SAMPLES = 100

FINAL_ADAPTER_DIR = os.path.join(
    MODEL_CHECKPOINT_DIR,
    "final_adapter"
)

if not os.path.isdir(FINAL_ADAPTER_DIR):
    raise FileNotFoundError(
        "Fine-tuned adapter not found. Run Cell 6 before Cell 7."
    )

# Load tokenizer
eval_tokenizer = AutoTokenizer.from_pretrained(
    FINAL_ADAPTER_DIR,
    token=HF_TOKEN
)

if eval_tokenizer.pad_token is None:
    eval_tokenizer.pad_token = eval_tokenizer.eos_token

eval_tokenizer.padding_side = "left"

# Reload the quantized base model
eval_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=eval_bnb_config,
    device_map="auto",
    torch_dtype=COMPUTE_DTYPE,
    token=HF_TOKEN
)

# Attach trained LoRA adapter
eval_model = PeftModel.from_pretrained(
    base_model,
    FINAL_ADAPTER_DIR
)

eval_model.eval()
eval_model.config.use_cache = True

print("Fine-tuned Gemma adapter loaded successfully.")


# Prepare validation data
eval_val_df = pd.read_csv(VAL_CSV).dropna(
    subset=["Abstract", "Highlights"]
).reset_index(drop=True)

eval_val_df["Abstract"] = eval_val_df["Abstract"].astype(str)
eval_val_df["Highlights"] = eval_val_df["Highlights"].astype(str)

if MAX_EVAL_SAMPLES is not None:
    eval_val_df = eval_val_df.head(MAX_EVAL_SAMPLES).copy()

sources = eval_val_df["Abstract"].tolist()
references = eval_val_df["Highlights"].tolist()


def build_prompt(abstract):
    messages = [
        {
            "role": "user",
            "content": (
                "Generate concise scientific research highlights "
                "for this abstract:\n" + str(abstract).strip()
            )
        }
    ]

    return eval_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


def generate_highlight(abstract):
    prompt = build_prompt(abstract)

    inputs = eval_tokenizer(
        prompt,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        return_tensors="pt"
    )

    input_device = eval_model.get_input_embeddings().weight.device
    inputs = {
        key: value.to(input_device)
        for key, value in inputs.items()
    }

    prompt_length = inputs["input_ids"].shape[1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    with torch.inference_mode():
        output = eval_model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=eval_tokenizer.pad_token_id,
            eos_token_id=eval_tokenizer.eos_token_id,
            use_cache=True
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start_time

    generated_ids = output[0][prompt_length:]

    prediction = eval_tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    generated_token_count = len(generated_ids)

    return prediction, elapsed, generated_token_count


# Warm-up generation
print("Warming up model...")

_ = generate_highlight(
    "This is a short warm-up scientific abstract."
)

# Validation benchmark
predictions = []
latencies = []
token_counts = []

print(
    f"\nEvaluating {len(sources)} validation samples..."
)

for index, abstract in enumerate(sources):
    prediction, elapsed, token_count = generate_highlight(
        abstract
    )

    predictions.append(prediction)
    latencies.append(elapsed)
    token_counts.append(token_count)

    if (index + 1) % 10 == 0 or index + 1 == len(sources):
        print(
            f"Sample {index + 1}/{len(sources)} | "
            f"Latency: {elapsed:.2f}s | "
            f"Generated tokens: {token_count}"
        )


# Calculate benchmark metrics
benchmark_results = evaluate_core_benchmarks(
    predictions=predictions,
    sources=sources,
    references=references,
    latencies=latencies,
    generated_tokens=token_counts
)

benchmark_results.update({
    "model": "gemma-2-2b",
    "model_id": MODEL_ID,
    "max_source_length": MAX_SOURCE_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "evaluation_samples": len(predictions),
    "fine_tuning": True,
    "fine_tuning_method": "4-bit QLoRA",
    "quantization": "NF4",
    "evaluation_type": "Fine-tuned validation inference"
})

# Save JSON benchmark results
benchmark_json = os.path.join(
    RESULTS_DIR,
    "benchmark_gemma-2-2b.json"
)

with open(benchmark_json, "w", encoding="utf-8") as file:
    json.dump(
        benchmark_results,
        file,
        indent=4,
        ensure_ascii=False
    )

# Save validation predictions
validation_output = eval_val_df.copy()
validation_output["Generated_Highlights"] = predictions
validation_output["Inference_Latency_Sec"] = latencies
validation_output["Generated_Tokens"] = token_counts

validation_csv = os.path.join(
    RESULTS_DIR,
    "gemma_validation_predictions.csv"
)

validation_output.to_csv(
    validation_csv,
    index=False
)

print("\nValidation benchmark completed.")
print("Benchmark JSON:", benchmark_json)
print("Validation predictions:", validation_csv)

print("\n--- GEMMA BENCHMARK RESULTS ---")

for key, value in benchmark_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")


# Generate predictions for masked test set
if test_df is not None:
    print(
        f"\nGenerating predictions for {len(test_df)} masked test samples..."
    )

    test_predictions = []

    for index, abstract in enumerate(test_df["Abstract"].tolist()):
        prediction, _, _ = generate_highlight(abstract)
        test_predictions.append(prediction)

        if (index + 1) % 25 == 0 or index + 1 == len(test_df):
            print(
                f"Test progress: {index + 1}/{len(test_df)}"
            )

    test_output = test_df.copy()
    test_output["Highlights"] = test_predictions

    test_output_csv = os.path.join(
        RESULTS_DIR,
        "MixSub-SciHigh_test_predictions_gemma-2-2b.csv"
    )

    test_output.to_csv(
        test_output_csv,
        index=False
    )

    print("Masked test predictions saved:", test_output_csv)

else:
    print(
        "Masked test predictions skipped because the test CSV is unavailable."
    )


# Cleanup
del eval_model
del base_model

gc.collect()
torch.cuda.empty_cache()

print("\nGemma-2-2B pipeline completed successfully.")
